# Final Hybrid Streamlit Deployment

Updated Colab notebook for the hybrid surveillance pipeline:

- **Binary anomaly detection:** existing binary ensemble
- **Activity:** OLD `clf_action_fine`
- **Weapon:** OLD `clf_weapon`
- **People:** NEW Temporal Adapter + Summary Fusion
- **Location:** NEW Temporal Adapter + Summary Fusion
- **Summary:** existing Q-Former + LoRA checkpoint

The notebook expects `app_core1.py` and `streamlit_app_v4.py` in GitHub. Existing V3 files remain untouched.

**Run all cells from top to bottom using a GPU runtime.**


In [1]:
# ============================================================
# HYBRID STREAMLIT SURVEILLANCE DEPLOYMENT
# ============================================================

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path


## 1. Configuration


In [2]:
REPO_DIR = Path("/content/MicroVLM_Edge1")
REPO_URL = "https://github.com/SudeepIITM/MicroVLM_Edge1.git"

APP = "streamlit_app_v4.py"
PORT = 8501

# Existing binary artifacts
BINARY_DRIVE_DIR = Path(
    "/content/drive/MyDrive/models_ucf_v2"
)

# NEW multitask temporal + summary-fusion artifacts
MULTITASK_DRIVE_DIR = Path(
    "/content/drive/MyDrive/Project_VLM/UCF_Multitask_Temporal"
)

# OLD classifier bundle used for Activity + Weapon
OLD_MULTICLASS_BUNDLE_DRIVE = Path(
    "/content/drive/MyDrive/models3/multiclass_bundle.pkl"
)

# Existing Q-Former + LoRA summarization checkpoint
SUMMARY_DRIVE_DIR = Path(
    "/content/drive/MyDrive/Project_VLM/"
    "ucf_qwen_v9_qformer/checkpoint-414"
)

DEPLOY_ROOT = Path("/content/deployment_models")

BINARY_LOCAL = DEPLOY_ROOT / "binary"
MULTITASK_LOCAL = DEPLOY_ROOT / "multitask"
OLD_MULTICLASS_LOCAL = DEPLOY_ROOT / "old_multiclass"
SUMMARY_LOCAL = DEPLOY_ROOT / "summary"

OLD_MULTICLASS_BUNDLE_LOCAL = (
    OLD_MULTICLASS_LOCAL / "multiclass_bundle.pkl"
)

STREAMLIT_LOG = Path("/content/streamlit.log")
CLOUDFLARED = Path("/content/cloudflared")

print("Configuration ready.")


Configuration ready.


## 2. Helper Functions


In [3]:
def run(command, cwd=None, check=True, capture=False, env=None):
    print("$", " ".join(map(str, command)))

    return subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        capture_output=capture,
        env=env,
    )


def require_path(path, description):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"{description} not found:\n{path}"
        )


def verify_files(folder, required_files, title):
    folder = Path(folder)

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    missing = []

    for filename in required_files:
        path = folder / filename
        status = "OK" if path.exists() else "MISSING"
        print(f"{filename:35s} : {status}")

        if not path.exists():
            missing.append(filename)

    if missing:
        raise FileNotFoundError(
            f"Missing required files in {folder}:\n"
            + "\n".join(f" - {name}" for name in missing)
        )


def copy_directory_contents(source, destination):
    source = Path(source)
    destination = Path(destination)

    require_path(source, "Source directory")

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    for item in source.iterdir():
        target = destination / item.name

        if item.is_dir():
            shutil.copytree(
                item,
                target,
                dirs_exist_ok=True,
            )
        else:
            shutil.copy2(item, target)


def stop_old_streamlit():
    subprocess.run(
        ["pkill", "-f", "streamlit run"],
        check=False,
    )


## 3. Mount Google Drive


In [5]:
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

print("Google Drive ready.")


Mounted at /content/drive
Google Drive ready.


## 4. Clone or Update Repository


In [6]:
from google.colab import userdata

github_token = userdata.get("Github")

if not REPO_DIR.exists():
    if github_token:
        authenticated_url = (
            f"https://{github_token}@github.com/"
            "SudeepIITM/MicroVLM_Edge1.git"
        )

        run([
            "git",
            "clone",
            authenticated_url,
            REPO_DIR,
        ])

        run([
            "git",
            "remote",
            "set-url",
            "origin",
            REPO_URL,
        ], cwd=REPO_DIR)

    else:
        run([
            "git",
            "clone",
            REPO_URL,
            REPO_DIR,
        ])

else:
    print("Repository already exists. Pulling latest changes.")

    pull_result = run(
        ["git", "pull"],
        cwd=REPO_DIR,
        check=False,
        capture=True,
    )

    print(pull_result.stdout)
    print(pull_result.stderr)

require_path(
    REPO_DIR / "app_core1.py",
    "Updated hybrid app_core1.py",
)

require_path(
    REPO_DIR / APP,
    f"Streamlit application {APP}",
)

print("Repository ready.")


$ git clone https://ghp_c@github.com/SudeepIITM/MicroVLM_Edge1.git /content/MicroVLM_Edge1
$ git remote set-url origin https://github.com/SudeepIITM/MicroVLM_Edge1.git
Repository ready.


## 5. Install Dependencies


In [7]:
requirements = REPO_DIR / "requirements.txt"

require_path(
    requirements,
    "requirements.txt",
)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    requirements,
])

# Ensure joblib is available for the old clf_* bundle.
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "joblib",
])

print("Dependencies installed.")


$ /usr/bin/python3 -m pip install -q -r /content/MicroVLM_Edge1/requirements.txt
$ /usr/bin/python3 -m pip install -q joblib
Dependencies installed.


## 6. Copy Hybrid Model Artifacts


In [8]:
print("\nPreparing deployment model folders.")

if DEPLOY_ROOT.exists():
    shutil.rmtree(DEPLOY_ROOT)

for folder in [
    BINARY_LOCAL,
    MULTITASK_LOCAL,
    OLD_MULTICLASS_LOCAL,
    SUMMARY_LOCAL,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("\nCopying binary artifacts...")
copy_directory_contents(
    BINARY_DRIVE_DIR,
    BINARY_LOCAL,
)

print("\nCopying NEW multitask artifacts...")
copy_directory_contents(
    MULTITASK_DRIVE_DIR,
    MULTITASK_LOCAL,
)

print("\nCopying OLD Activity/Weapon classifier bundle...")
require_path(
    OLD_MULTICLASS_BUNDLE_DRIVE,
    "Old multiclass_bundle.pkl",
)

shutil.copy2(
    OLD_MULTICLASS_BUNDLE_DRIVE,
    OLD_MULTICLASS_BUNDLE_LOCAL,
)

print("\nCopying summarization checkpoint...")
copy_directory_contents(
    SUMMARY_DRIVE_DIR,
    SUMMARY_LOCAL,
)

print("\nAll deployment artifacts copied.")



Preparing deployment model folders.

Copying binary artifacts...

Copying NEW multitask artifacts...

Copying OLD Activity/Weapon classifier bundle...

Copying summarization checkpoint...

All deployment artifacts copied.


## 7. Verify Hybrid Model Files


In [9]:
BINARY_REQUIRED = [
    "simple_adapter.pt",
]

MULTITASK_REQUIRED = [
    "temporal_adapter.pkl",
    "summary_projector.pkl",
    "fusion.pkl",
    "activity_head.pkl",
    "weapon_head.pkl",
    "people_head.pkl",
    "location_head.pkl",
    "label_encoders.pkl",
    "multitask_temporal_model.pkl",
    "training_metadata.json",
]

SUMMARY_REQUIRED = [
    "adapter_config.json",
]

verify_files(
    BINARY_LOCAL,
    BINARY_REQUIRED,
    "BINARY MODEL ARTIFACTS",
)

verify_files(
    MULTITASK_LOCAL,
    MULTITASK_REQUIRED,
    "NEW PEOPLE / LOCATION MODEL ARTIFACTS",
)

verify_files(
    OLD_MULTICLASS_LOCAL,
    ["multiclass_bundle.pkl"],
    "OLD ACTIVITY / WEAPON CLASSIFIER BUNDLE",
)

verify_files(
    SUMMARY_LOCAL,
    SUMMARY_REQUIRED,
    "SUMMARY CHECKPOINT",
)

print("\nHybrid model verification passed.")



BINARY MODEL ARTIFACTS
simple_adapter.pt                   : OK

NEW PEOPLE / LOCATION MODEL ARTIFACTS
temporal_adapter.pkl                : OK
summary_projector.pkl               : OK
fusion.pkl                          : OK
activity_head.pkl                   : OK
weapon_head.pkl                     : OK
people_head.pkl                     : OK
location_head.pkl                   : OK
label_encoders.pkl                  : OK
multitask_temporal_model.pkl        : OK
training_metadata.json              : OK

OLD ACTIVITY / WEAPON CLASSIFIER BUNDLE
multiclass_bundle.pkl               : OK

SUMMARY CHECKPOINT
adapter_config.json                 : OK

Hybrid model verification passed.


## 8. Inspect Old Activity and Weapon Bundle


In [10]:
import joblib

old_bundle = joblib.load(
    OLD_MULTICLASS_BUNDLE_LOCAL
)

required_old_keys = [
    "clf_action_fine",
    "le_action_fine",
    "clf_weapon",
    "le_weapon",
]

print("Required old classifier keys:")

for key in required_old_keys:
    status = "OK" if key in old_bundle else "MISSING"
    print(f"{key:30s} : {status}")

missing = [
    key
    for key in required_old_keys
    if key not in old_bundle
]

if missing:
    raise KeyError(
        "Old multiclass bundle is missing: "
        + ", ".join(missing)
    )

print("\nOLD Activity + Weapon bundle is valid.")


Required old classifier keys:
clf_action_fine                : OK
le_action_fine                 : OK
clf_weapon                     : OK
le_weapon                      : OK

OLD Activity + Weapon bundle is valid.


## 9. Set Hybrid Deployment Environment


In [11]:
# Existing binary model
os.environ["MODEL_DIR"] = str(BINARY_LOCAL)
os.environ["BINARY_MODEL_DIR"] = str(BINARY_LOCAL)

# NEW model used for People + Location
os.environ["MULTITASK_MODEL_DIR"] = str(
    MULTITASK_LOCAL
)

os.environ["MULTITASK_MODEL_PATH"] = str(
    MULTITASK_LOCAL /
    "multitask_temporal_model.pkl"
)

os.environ["MULTITASK_METADATA_PATH"] = str(
    MULTITASK_LOCAL /
    "training_metadata.json"
)

os.environ["MULTITASK_ENCODERS_PATH"] = str(
    MULTITASK_LOCAL /
    "label_encoders.pkl"
)

# OLD classifier bundle used for Activity + Weapon
os.environ["OLD_MULTICLASS_BUNDLE_PATH"] = str(
    OLD_MULTICLASS_BUNDLE_LOCAL
)

# Existing Q-Former + LoRA summarization
os.environ["SUMMARY_CHECKPOINT"] = str(
    SUMMARY_LOCAL
)

os.environ["LORA_CHECKPOINT"] = str(
    SUMMARY_LOCAL
)

os.environ["NUM_FRAMES"] = "8"
os.environ["STREAMLIT_SERVER_PORT"] = str(PORT)

print("\n" + "=" * 80)
print("HYBRID DEPLOYMENT CONFIGURATION")
print("=" * 80)

for key in [
    "BINARY_MODEL_DIR",
    "MULTITASK_MODEL_DIR",
    "MULTITASK_MODEL_PATH",
    "MULTITASK_METADATA_PATH",
    "MULTITASK_ENCODERS_PATH",
    "OLD_MULTICLASS_BUNDLE_PATH",
    "SUMMARY_CHECKPOINT",
]:
    print(f"{key:32s} = {os.environ[key]}")

print("\nTask mapping:")
print("  Activity -> OLD clf_action_fine")
print("  Weapon   -> OLD clf_weapon")
print("  People   -> NEW Temporal + Summary Fusion")
print("  Location -> NEW Temporal + Summary Fusion")



HYBRID DEPLOYMENT CONFIGURATION
BINARY_MODEL_DIR                 = /content/deployment_models/binary
MULTITASK_MODEL_DIR              = /content/deployment_models/multitask
MULTITASK_MODEL_PATH             = /content/deployment_models/multitask/multitask_temporal_model.pkl
MULTITASK_METADATA_PATH          = /content/deployment_models/multitask/training_metadata.json
MULTITASK_ENCODERS_PATH          = /content/deployment_models/multitask/label_encoders.pkl
OLD_MULTICLASS_BUNDLE_PATH       = /content/deployment_models/old_multiclass/multiclass_bundle.pkl
SUMMARY_CHECKPOINT               = /content/deployment_models/summary

Task mapping:
  Activity -> OLD clf_action_fine
  Weapon   -> OLD clf_weapon
  People   -> NEW Temporal + Summary Fusion
  Location -> NEW Temporal + Summary Fusion


## 10. Validate Hybrid `app_core.py`


In [12]:
app_core_path = REPO_DIR / "app_core1.py"

app_core_text = app_core_path.read_text(
    encoding="utf-8"
)

required_markers = [
    "OLD_MULTICLASS_BUNDLE_PATH",
    "clf_action_fine",
    "clf_weapon",
    "old_mc_bundle",
]

print("Checking hybrid app_core1.py markers:")

missing_markers = []

for marker in required_markers:
    status = "OK" if marker in app_core_text else "MISSING"
    print(f"{marker:35s} : {status}")

    if marker not in app_core_text:
        missing_markers.append(marker)

if missing_markers:
    raise RuntimeError(
        "GitHub app_core.py is not the hybrid version. "
        "Replace it with app_core_hybrid_old_activity_weapon.py "
        "and rename it to app_core.py. Missing markers: "
        + ", ".join(missing_markers)
    )

print("\nHybrid app_core1.py confirmed.")


Checking hybrid app_core1.py markers:
OLD_MULTICLASS_BUNDLE_PATH          : OK
clf_action_fine                     : OK
clf_weapon                          : OK
old_mc_bundle                       : OK

Hybrid app_core1.py confirmed.


## 11. Download Cloudflared


In [13]:
if not CLOUDFLARED.exists():
    run([
        "wget",
        "-q",
        "https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64",
        "-O",
        CLOUDFLARED,
    ])

run([
    "chmod",
    "+x",
    CLOUDFLARED,
])

run([
    CLOUDFLARED,
    "--version",
])


$ wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
$ chmod +x /content/cloudflared
$ /content/cloudflared --version


CompletedProcess(args=['/content/cloudflared', '--version'], returncode=0)

## 12. Start Streamlit


In [14]:
stop_old_streamlit()
time.sleep(2)

log_handle = open(
    STREAMLIT_LOG,
    "w",
)

streamlit_command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    APP,
    "--server.port",
    str(PORT),
    "--server.headless",
    "true",
    "--server.enableCORS",
    "false",
    "--server.enableXsrfProtection",
    "false",
]

print("\nStarting Streamlit hybrid deployment...")

streamlit_process = subprocess.Popen(
    streamlit_command,
    cwd=str(REPO_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("Waiting for Streamlit to initialize...")

server_ready = False

for second in range(1, 121):
    time.sleep(1)

    if streamlit_process.poll() is not None:
        break

    try:
        import urllib.request

        response = urllib.request.urlopen(
            f"http://127.0.0.1:{PORT}",
            timeout=2,
        )

        if response.status == 200:
            server_ready = True
            print(
                f"Streamlit ready after {second} seconds."
            )
            break

    except Exception:
        pass

if not server_ready:
    print("\nStreamlit did not become ready.")
    print("\nLast 100 log lines:\n")

    if STREAMLIT_LOG.exists():
        lines = STREAMLIT_LOG.read_text(
            errors="replace"
        ).splitlines()

        print("\n".join(lines[-100:]))

    raise RuntimeError(
        "Streamlit startup failed. "
        "Check the log above."
    )

print("\nStreamlit hybrid deployment is ready.")



Starting Streamlit hybrid deployment...
Waiting for Streamlit to initialize...
Streamlit ready after 3 seconds.

Streamlit hybrid deployment is ready.


## 13. Start Public Cloudflare Tunnel


In [ ]:
print("\n" + "=" * 80)
print("STARTING PUBLIC CLOUDFLARE TUNNEL")
print("=" * 80)

print(
    "The public *.trycloudflare.com URL "
    "will appear below.\n"
)

tunnel_command = [
    str(CLOUDFLARED),
    "tunnel",
    "--url",
    f"http://127.0.0.1:{PORT}",
]

cloudflare_process = None

try:
    cloudflare_process = subprocess.Popen(
        tunnel_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    print("Cloudflare tunnel process started...\n")

    for line in cloudflare_process.stdout:
        print(line, end="", flush=True)

except KeyboardInterrupt:
    print("\nStopping deployment...")

except Exception as e:
    print(f"\nCloudflare tunnel error: {e}")

finally:

    if cloudflare_process is not None:
        if cloudflare_process.poll() is None:
            cloudflare_process.terminate()

    if streamlit_process.poll() is None:
        streamlit_process.terminate()

    log_handle.close()

    print("Deployment stopped.")


STARTING PUBLIC CLOUDFLARE TUNNEL
The public *.trycloudflare.com URL will appear below.

Cloudflare tunnel process started...

2026-07-16T02:19:18Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-16T02:19:18Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-16T02:19:24Z INF +--------------------------------------------------------------------------------------------+
2026-07-16T02:19:24Z INF |  Your quick Tunnel has been created! Visit it 

## Final Hybrid Mapping

| Output | Model |
|---|---|
| Binary Normal / Anomalous | Existing binary ensemble |
| Activity | OLD `clf_action_fine` |
| Weapon | OLD `clf_weapon` |
| People | NEW Temporal Adapter + Summary Fusion |
| Location | NEW Temporal Adapter + Summary Fusion |
| Video Summary | Existing Q-Former + LoRA |

The notebook validates that the GitHub `app_core.py` contains the hybrid classifier logic before starting Streamlit.
